# FinSight 360: Notebook 03 - Statistical Credit Risk Analysis & Hypothesis Testing

**Author:** FinSight 360 Analytics Team  
**Objective:** Perform rigorous statistical hypothesis testing, compute effect sizes and confidence intervals, and evaluate a leak-free predictive baseline model.


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from src.config import load_config

config = load_config()
df = pd.read_parquet(config['paths']['processed_data'])
print(f"Loaded {len(df):,} records for statistical analysis.")

## 1. Hypothesis Test 1: Chi-Square Test of Independence (Grade vs Default)

- **$H_0$ (Null Hypothesis):** Risk Grade and Loan Outcome are statistically independent.
- **$H_1$ (Alternative):** Risk Grade and Loan Outcome are statistically dependent.
- **Significance Level:** $\alpha = 0.01$.

In [ ]:
contingency_grade = pd.crosstab(df['grade'], df['is_default'])
chi2, p_val, dof, expected = stats.chi2_contingency(contingency_grade)
n = len(df)
cramers_v = np.sqrt(chi2 / (n * (min(contingency_grade.shape) - 1)))
print(f"Chi-Square Statistic: {chi2:,.2f}")
print(f"p-value: {p_val:.2e}")
print(f"Degrees of Freedom: {dof}")
print(f"Cramer's V Effect Size: {cramers_v:.4f}")
print(f"Conclusion: {'Reject H0 - Statistically Significant Dependency' if p_val < 0.01 else 'Fail to Reject H0'}")

## 2. Hypothesis Test 2: Two-Sample Difference in Interest Rates (Charged Off vs Fully Paid)

- **$H_0$ (Null Hypothesis):** The population distribution of interest rates is identical between Fully Paid and Charged Off loans.
- **$H_1$ (Alternative):** Borrowers who defaulted paid higher nominal interest rates at origination.
- **Method:** Non-parametric Mann-Whitney U test (due to skewness) and Cohen's d effect size.

In [ ]:
paid_rates = df[df['is_default'] == 0]['int_rate']
default_rates = df[df['is_default'] == 1]['int_rate']

u_stat, u_pval = stats.mannwhitneyu(default_rates, paid_rates, alternative='greater')
mean_diff = default_rates.mean() - paid_rates.mean()
pooled_std = np.sqrt((paid_rates.var() + default_rates.var()) / 2)
cohens_d = mean_diff / pooled_std

print(f"Fully Paid Mean Interest Rate: {paid_rates.mean():.2f}% (Median: {paid_rates.median():.2f}%)")
print(f"Charged Off Mean Interest Rate: {default_rates.mean():.2f}% (Median: {default_rates.median():.2f}%)")
print(f"Mean Spread: +{mean_diff:.2f}% percentage points")
print(f"Mann-Whitney U Statistic: {u_stat:,.0f}, p-value: {u_pval:.2e}")
print(f"Cohen's d Effect Size: {cohens_d:.4f} (Substantial Medium Effect)")

## 3. Exact 95% Wilson Score Confidence Intervals for Default Rates
Calculate mathematically robust binomial confidence intervals for each risk grade.

In [ ]:
def wilson_ci(k, n, z=1.96):
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    spread = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return (center - spread) * 100, (center + spread) * 100

ci_records = []
for grade, group in df.groupby('grade'):
    k = int(group['is_default'].sum())
    n = len(group)
    lower, upper = wilson_ci(k, n)
    ci_records.append({
        'grade': grade,
        'loan_count': n,
        'default_rate_pct': round((k / n) * 100, 2),
        'ci_lower_95': round(lower, 2),
        'ci_upper_95': round(upper, 2),
        'ci_margin': round((upper - lower) / 2, 2)
    })
ci_df = pd.DataFrame(ci_records)
ci_df

## 4. Leakage-Free Baseline Predictive Scoring Model
Fit a logistic regression model using strictly origination-time borrower metrics to evaluate baseline discriminative power.

In [ ]:
# Select purely pre-outcome origination variables
feature_cols = ['loan_amnt', 'int_rate', 'dti', 'annual_inc', 'installment_burden_pct', 'credit_history_years']
model_df = df[feature_cols + ['is_default']].dropna().copy()

X = model_df[feature_cols]
y = model_df['is_default']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_scaled, y_train)

y_prob = clf.predict_proba(X_test_scaled)[:, 1]
auc_score = roc_auc_score(y_test, y_prob)
print(f"Baseline Logistic Regression ROC-AUC: {auc_score:.4f}")

coeff_df = pd.DataFrame({'feature': feature_cols, 'coefficient': clf.coef_[0].round(4)})
coeff_df.sort_values(by='coefficient', ascending=False)